# 02 — Assumption composition: scientific stances as constraints

Partial identification is only as credible as the assumptions you state.
**partialid** treats those assumptions as composable objects: each literature
idea is an atom (or a thin recipe), and `&` takes their intersection.

This notebook reuses the job-training extract from notebook 01 and compares
intervals under different **scientific stances** — without introducing graphs
or sensitivity *paths* yet.


## Reload the study (self-contained setup)


In [1]:
from pathlib import Path

def tutorial_data(name: str) -> Path:
    """Resolve notebooks/data/<name> whether __file__ is set (CI) or not."""
    here = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
    candidates = [
        here / "data" / name,
        here / "notebooks" / "data" / name,
        Path.cwd() / "notebooks" / "data" / name,
        Path.cwd() / "data" / name,
    ]
    for p in candidates:
        if p.is_file():
            return p
    raise FileNotFoundError(f"could not find notebooks/data/{name}; tried {candidates}")

import pandas as pd
from partialid import Assumptions, Query, bound, from_dataframe

df = pd.read_csv(tutorial_data("job_training.csv"))
data = from_dataframe(df, columns=["treated", "employed"])
Q = Query.interventional(outcome="employed", do={"treated": 1}, value=1)
print(f"N={len(df)}  |  {Q.describe()}")
print(df.groupby("treated")["employed"].mean().rename("employment rate"))


N=2000  |  P(employed=1 | do(treated=1))
treated
0    0.404762
1    0.712702
Name: employment rate, dtype: float64


## Two families of assumptions

1. **Structural / qualitative** — order, diagrams, exclusions, MTR/MTS,
   instruments, hard independence. These carve faces of the response-type set.
2. **Sensitivity / quantitative budgets** — Tan MSM \(\Lambda\), Rosenbaum
   \(\Gamma\), \(c\)-dependence, margin budgets, \(f\)-sensitivity. These
   dial *how much* confounding you allow.

`&` is a meet: adding assumptions can only shrink (or leave unchanged) the
identified interval.


## Structural faces side-by-side


In [2]:
base = Assumptions.order("treated", "employed")
faces = {
    "Manski / order only": Assumptions.manski("treated", "employed"),
    "MTR (program never hurts)": base & Assumptions.mtr("employed", "treated"),
    "MTS (selection monotone)": base & Assumptions.mts("treated", "employed"),
    "MTR + MTS": base & Assumptions.mtr("employed", "treated") & Assumptions.mts("treated", "employed"),
}
print(f"{'stance':28s}  {'interval':18s}  width   tight")
for name, a in faces.items():
    r = bound(data.variables, data.P, Q, a)
    print(f"{name:28s}  [{r.lower:.4f}, {r.upper:.4f}]  {r.width:.4f}  {r.tight}")


stance                        interval            width   tight
Manski / order only           [0.3535, 0.8575]  0.5040  True
MTR (program never hurts)     [0.5575, 0.8575]  0.3000  True
MTS (selection monotone)      [0.3535, 0.7127]  0.3592  True
MTR + MTS                     [0.5575, 0.7127]  0.1552  True


In the job-training story:

- **Manski** refuses almost all structure beyond temporal order — wide bounds.
- **MTR** says the program is not harmful for any latent type.
- **MTS** restricts how latent employability drives selection into training.
- Combining both is a stronger joint claim; check that you believe *each* piece.


## Fixed sensitivity budgets

Quantitative budgets answer: “if confounding is limited in *this* way, how
tight is the interval?” Below we compare common budgets at **one** defended
value each, always on top of the Manski order.


In [3]:
budgets = {
    "Tan MSM Λ=2": base & Assumptions.tan_msm("treated", 2.0),
    "Rosenbaum Γ=4": base & Assumptions.rosenbaum_gamma("treated", Gamma=4.0),
    "c-dependence c=0.1": base & Assumptions.c_dependence("treated", 0.1),
    "margin κ=0.05": base & Assumptions.margin_budget(0.05),
}
print(f"{'budget':22s}  {'interval / status':32s}  method")
for name, a in budgets.items():
    r = bound(data.variables, data.P, Q, a)
    if r.success:
        status = f"[{r.lower:.4f}, {r.upper:.4f}] w={r.width:.4f}"
    else:
        status = f"infeasible ({r.message})"
    print(f"{name:22s}  {status:32s}  {r.method}")


budget                  interval / status                 method
Tan MSM Λ=2             [0.5679, 0.7851] w=0.2172         msm_agg/msm_margin_binary_xy
Rosenbaum Γ=4           [0.5679, 0.7851] w=0.2172         msm_agg/msm_margin_binary_xy_gamma
c-dependence c=0.1      [0.6402, 0.7609] w=0.1208         msm_agg/msm_transport_lp
margin κ=0.05           [0.6627, 0.7627] w=0.1000         lp


Reading the knobs (intuition, not proofs):

- **Tan MSM \(\Lambda\)** — odds of treatment may differ by at most \(\Lambda\)
  across latent strata (\(\Lambda=1\) is no unobserved confounding on this margin).
- **Rosenbaum \(\Gamma\)** — related odds-ratio sensitivity (anchored form).
- **\(c\)-dependence** — absolute deviation of propensity from a baseline.
- **Margin \(\kappa\)** — soft independence / free-mass deviation from a
  reference factorization.


## Recipes are thin wrappers


In [4]:
recipe = Assumptions.sensitivity_msm("treated", "employed", 3.0)
atoms = Assumptions.manski("treated", "employed") & Assumptions.tan_msm("treated", 3.0)
r1 = bound(data.variables, data.P, Q, recipe)
r2 = bound(data.variables, data.P, Q, atoms)
print("recipe describe:")
for line in recipe.describe():
    print(" ", line)
print(f"widths equal? {round(r1.width, 6)} == {round(r2.width, 6)}")


recipe describe:
  OrderAssumption: OrderAssumption(order=('treated', 'employed'))
  TanMSMAssumption: TanMSMAssumption(treatment='treated', Lambda=3.0, anchored=True)
widths equal? 0.336 == 0.336


Prefer **atoms + `&`** when a recipe’s defaults do not match your setting
(e.g. you already have MTR and only want to add MSM).


## Advanced atoms (short)

Two less common tools: a hard **support** cap on the number of positive
response types (MIP), and **\(f\)-sensitivity** (divergence ball; uses CVXPY).
Reach for these when a referee asks for sparsity or a specific divergence
sensitivity model — not as day-one defaults.


In [5]:
for name, a in [
    ("support ≤ 4 types", base & Assumptions.support_bound(4)),
    ("f-sensitivity TV ρ=0.05", base & Assumptions.f_sensitivity("treated", 0.05, f="tv")),
]:
    r = bound(data.variables, data.P, Q, a)
    status = f"[{r.lower:.4f}, {r.upper:.4f}] w={r.width:.4f}" if r.success else f"fail: {r.message}"
    print(f"{name:28s}  {status}  method={r.method}")


support ≤ 4 types             [0.3535, 0.8575] w=0.5040  method=mip
f-sensitivity TV ρ=0.05       [0.6623, 0.7631] w=0.1008  method=convex


## When assumptions conflict with the data

If the observational law is incompatible with your assumption set, `bound`
returns `success=False`. Always read `message` — do not interpret an empty or
failed solve as “effect is zero.”


In [6]:
# Extreme support cap: often infeasible for rich binary tables
r_bad = bound(data.variables, data.P, Q, base & Assumptions.support_bound(1))
print("success:", r_bad.success)
print("message:", r_bad.message)


success: False
message: lo:The problem is infeasible. (HiGHS Status 8: model_status is Infeasible; primal_status is None); hi:The problem is infeasible. (HiGHS Status 8: model_status is Infeasible; primal_status is None)


## Decision guide

1. Start with **Manski** to see the worst-case width.
2. Add **structural faces** you actually believe (MTR, MTS, exclusions, IV…).
3. Add a **sensitivity budget** you can defend to a skeptic.
4. Report the interval **and** `a.describe()` — the assumptions are part of the result.

**Next:** [03_sensitivity_paths.ipynb](03_sensitivity_paths.ipynb) — vary the
budget continuously with `bound_path` and find breakdown points.
